# 🌫️ VayuSangam — XGBoost AQI Model Training

Trains a real XGBoost regressor to predict Delhi NCR **PM2.5** and **AQI** from:
- Historical weather (Open-Meteo — **free, no key**)
- Historical fire activity (NASA FIRMS — uses your existing key)
- Ground truth AQI (Kaggle: Air Quality Data in India — **free download**)

**Output:** `xgb_pm25.joblib` + `xgb_aqi.joblib` → drop in `backend/data/ml/`

**Runtime:** CPU is fine. Training ~3–5 min.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q xgboost shap joblib scikit-learn pandas numpy requests kaggle

## Step 2 — Kaggle Setup
1. Go to https://kaggle.com → Your Profile → Settings → API → **Create New Token**
2. Upload `kaggle.json` below
3. OR skip and manually upload the CSV in Step 3B

In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()  # upload kaggle.json
os.makedirs('/root/.config/kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials set')

## Step 3A — Download Ground Truth AQI Data (Kaggle API)

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)
!kaggle datasets download -d rohanrao/air-quality-data-in-india -p data/raw --unzip
print('Files:', os.listdir('data/raw'))

## Step 3B — Alternative: Manual Upload
Download from https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india  
Then uncomment and run this cell:

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # upload city_hour.csv
# import shutil, os
# os.makedirs('data/raw', exist_ok=True)
# for fname in uploaded: shutil.move(fname, f'data/raw/{fname}')
print('(Using Kaggle API download - skip this cell)')

## Step 4 — Load & Filter Ground Truth (Delhi NCR only)

In [ ]:
import pandas as pd
import numpy as np

try:
    df_raw = pd.read_csv('data/raw/city_hour.csv', parse_dates=['Datetime'])
    df_raw.rename(columns={'Datetime':'timestamp','PM2.5':'pm25','AQI':'aqi','City':'city'}, inplace=True)
    print(f'Loaded city_hour.csv: {len(df_raw)} rows')
except FileNotFoundError:
    df_raw = pd.read_csv('data/raw/station_hour.csv', parse_dates=['Datetime'])
    df_raw.rename(columns={'Datetime':'timestamp','PM2.5':'pm25','AQI':'aqi'}, inplace=True)
    df_raw['city'] = 'Delhi'
    print(f'Loaded station_hour.csv: {len(df_raw)} rows')

# Filter Delhi NCR
delhi_keywords = ['Delhi','Gurugram','Noida','Faridabad','Ghaziabad']
mask = df_raw['city'].str.contains('|'.join(delhi_keywords), case=False, na=False)
df_delhi = df_raw[mask].copy()
print(f'Delhi NCR rows: {len(df_delhi)}')

# Hourly aggregate across stations
df_aqi = (
    df_delhi.groupby('timestamp')[['pm25','aqi']]
    .mean().reset_index().dropna(subset=['pm25'])
)
df_aqi['timestamp'] = pd.to_datetime(df_aqi['timestamp'])
df_aqi = df_aqi.set_index('timestamp').sort_index()

print(f'Clean hourly rows: {len(df_aqi)}')
print(f'Date range: {df_aqi.index.min()} → {df_aqi.index.max()}')
df_aqi.head()

## Step 5 — Fetch Historical Weather (Open-Meteo — free, no key)

In [ ]:
import requests, time

LAT, LON = 28.6139, 77.2090  # Delhi
start_date = df_aqi.index.min().strftime('%Y-%m-%d')
end_date   = df_aqi.index.max().strftime('%Y-%m-%d')
print(f'Fetching weather: {start_date} → {end_date}')

def fetch_openmeteo(lat, lon, start, end):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'hourly': ['temperature_2m','relative_humidity_2m','wind_speed_10m',
                   'wind_direction_10m','precipitation','boundary_layer_height'],
        'timezone': 'Asia/Kolkata',
        'wind_speed_unit': 'ms'
    }
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f'Retry {attempt+1}: {e}')
            time.sleep(5)
    raise RuntimeError('Open-Meteo failed')

all_weather = []
years = list(range(int(start_date[:4]), int(end_date[:4]) + 1))
for year in years:
    s = max(f'{year}-01-01', start_date)
    e = min(f'{year}-12-31', end_date)
    if s > e: continue
    print(f'  {s} → {e}...')
    data = fetch_openmeteo(LAT, LON, s, e)
    df_w = pd.DataFrame(data['hourly'])
    df_w['timestamp'] = pd.to_datetime(df_w['time'])
    df_w.drop(columns=['time'], inplace=True)
    all_weather.append(df_w)
    time.sleep(1)

df_weather = pd.concat(all_weather).set_index('timestamp').sort_index()
df_weather.rename(columns={
    'temperature_2m':'temperature_c',
    'relative_humidity_2m':'relative_humidity_pct',
    'wind_speed_10m':'wind_speed_mps',
    'wind_direction_10m':'wind_dir_deg',
    'boundary_layer_height':'pbl_height_m'
}, inplace=True)
print(f'\n✅ Weather rows: {len(df_weather)}')
df_weather.head()

## Step 6 — Add Fire Features (NASA FIRMS archive)

In [ ]:
from datetime import datetime, timedelta
from io import StringIO

FIRMS_MAP_KEY = 'f6a3860f7a9398edd089916711822612'
AREA = '73,27,80,31'

def fetch_firms_archive(key, area, start_date, end_date):
    all_dfs = []
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    chunk = 10
    while current <= end:
        chunk_end = min(current + timedelta(days=chunk-1), end)
        days = (chunk_end - current).days + 1
        url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/VIIRS_SNPP_SP/{area}/{days}/{current.strftime("%Y-%m-%d")}'
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and 'latitude' in r.text:
                df_f = pd.read_csv(StringIO(r.text))
                if len(df_f) > 0:
                    df_f['date'] = pd.to_datetime(df_f['acq_date'])
                    all_dfs.append(df_f[['date','frp']])
        except: pass
        current += timedelta(days=chunk)
        time.sleep(0.3)

    full_dates = pd.DataFrame({'date': pd.date_range(start_date, end_date, freq='D')})
    if not all_dfs:
        print('⚠️  FIRMS: no data — using zeros')
        return full_dates.assign(fire_count_24h=0, fire_frp_sum_24h=0.0, fire_frp_max_24h=0.0)

    df_fires = pd.concat(all_dfs)
    daily = df_fires.groupby('date')['frp'].agg(
        fire_count_24h='count', fire_frp_sum_24h='sum', fire_frp_max_24h='max'
    ).reset_index()
    return full_dates.merge(daily, on='date', how='left').fillna(0)

print('Fetching FIRMS (may take a few minutes)...')
df_fires_daily = fetch_firms_archive(
    FIRMS_MAP_KEY, AREA,
    df_aqi.index.min().strftime('%Y-%m-%d'),
    df_aqi.index.max().strftime('%Y-%m-%d')
)
print(f'✅ Fire data: {len(df_fires_daily)} days, max fires/day: {df_fires_daily["fire_count_24h"].max()}')

## Step 7 — Build Final Feature Dataset

In [ ]:
# Merge weather + AQI
df = df_weather.join(df_aqi, how='inner')
print(f'After weather+AQI join: {len(df)} rows')

# Expand fire data (daily → hourly via ffill)
df_fires_h = (df_fires_daily
    .assign(timestamp=lambda x: pd.to_datetime(x['date']))
    .set_index('timestamp').drop(columns=['date'])
    .resample('h').ffill())
df = df.join(df_fires_h, how='left')
df[['fire_count_24h','fire_frp_sum_24h','fire_frp_max_24h']] = \
    df[['fire_count_24h','fire_frp_sum_24h','fire_frp_max_24h']].fillna(0)

# Time features
df['hour_of_day']      = df.index.hour
df['month']            = df.index.month
df['day_of_week']      = df.index.dayofweek
df['is_stubble_season']= df['month'].isin([10,11]).astype(int)

# Lag features (past PM2.5 is very predictive!)
df['pm25_lag1h']  = df['pm25'].shift(1)
df['pm25_lag3h']  = df['pm25'].shift(3)
df['pm25_lag24h'] = df['pm25'].shift(24)

df.dropna(inplace=True)
print(f'\n✅ Final dataset: {len(df)} rows')
print(f'Columns: {list(df.columns)}')
df.describe()

## Step 8 — Temporal Train/Val/Test Split (no data leakage)

In [ ]:
FEATURES = [
    'temperature_c','relative_humidity_pct','wind_speed_mps','wind_dir_deg',
    'precipitation','pbl_height_m',
    'hour_of_day','month','day_of_week','is_stubble_season',
    'fire_count_24h','fire_frp_sum_24h','fire_frp_max_24h',
    'pm25_lag1h','pm25_lag3h','pm25_lag24h'
]
FEATURES = [c for c in FEATURES if c in df.columns]  # guard missing cols

n = len(df)
df_train = df.iloc[:int(n*0.80)]
df_val   = df.iloc[int(n*0.80):int(n*0.90)]
df_test  = df.iloc[int(n*0.90):]

X_train, y_pm25_train, y_aqi_train = df_train[FEATURES], df_train['pm25'], df_train['aqi']
X_val,   y_pm25_val,   y_aqi_val   = df_val[FEATURES],   df_val['pm25'],   df_val['aqi']
X_test,  y_pm25_test,  y_aqi_test  = df_test[FEATURES],  df_test['pm25'],  df_test['aqi']

print(f'Train: {len(df_train):5d} rows  {df_train.index.min().date()} → {df_train.index.max().date()}')
print(f'Val:   {len(df_val):5d} rows  {df_val.index.min().date()} → {df_val.index.max().date()}')
print(f'Test:  {len(df_test):5d} rows  {df_test.index.min().date()} → {df_test.index.max().date()}')
print(f'\nFeatures ({len(FEATURES)}):', FEATURES)

## Step 9 — Train XGBoost Models

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

PARAMS = dict(
    n_estimators=1000, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='reg:squarederror',
    early_stopping_rounds=50, eval_metric='rmse',
    n_jobs=-1, verbosity=1
)

print('Training PM2.5 model...')
xgb_pm25 = XGBRegressor(**PARAMS)
xgb_pm25.fit(X_train, y_pm25_train, eval_set=[(X_val, y_pm25_val)], verbose=100)

print('\nTraining AQI model...')
xgb_aqi = XGBRegressor(**PARAMS)
xgb_aqi.fit(X_train, y_aqi_train, eval_set=[(X_val, y_aqi_val)], verbose=100)

print('\n✅ Training complete!')

## Step 10 — Evaluate on Test Set

In [ ]:
import matplotlib.pyplot as plt

def evaluate(model, X, y, name):
    p = model.predict(X)
    r2 = r2_score(y, p)
    mae = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    print(f'  {name}: R²={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}')
    return p, r2, mae, rmse

print('📊 Test Set Results:')
p_pm25, r2_pm25, mae_pm25, rmse_pm25 = evaluate(xgb_pm25, X_test, y_pm25_test, 'PM2.5 (µg/m³)')
p_aqi,  r2_aqi,  mae_aqi,  rmse_aqi  = evaluate(xgb_aqi,  X_test, y_aqi_test,  'AQI')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, preds, y_true, title in [
    (axes[0], p_pm25, y_pm25_test, f'PM2.5  R²={r2_pm25:.3f}'),
    (axes[1], p_aqi,  y_aqi_test,  f'AQI    R²={r2_aqi:.3f}'),
]:
    ax.scatter(y_true, preds, alpha=0.25, s=4, c='steelblue')
    lims = [min(y_true.min(), preds.min()), max(y_true.max(), preds.max())]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted'); ax.set_title(title)
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — SHAP Explainability

In [ ]:
import shap
shap.initjs()

explainer = shap.TreeExplainer(xgb_pm25)
sample = X_test.sample(min(300, len(X_test)), random_state=42)
shap_values = explainer.shap_values(sample)

print(f'Base value (mean PM2.5): {explainer.expected_value:.2f} µg/m³')

plt.figure()
shap.summary_plot(shap_values, sample, feature_names=FEATURES, show=False)
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Show SHAP for one sample row (what the API will return)
row = X_test.iloc[[0]]
sv = explainer.shap_values(row)[0]
print('\nSHAP breakdown for one prediction:')
for feat, val in sorted(zip(FEATURES, sv), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {"▲" if val>0 else "▼"} {feat:30s}: {val:+.3f}')

## Step 12 — Save & Download Model Files

In [ ]:
import joblib, json
os.makedirs('model_output', exist_ok=True)

joblib.dump({'model': xgb_pm25, 'feature_names': FEATURES}, 'model_output/xgb_pm25.joblib')
joblib.dump({'model': xgb_aqi,  'feature_names': FEATURES}, 'model_output/xgb_aqi.joblib')

metadata = {
    'feature_names': FEATURES,
    'base_value_pm25': float(explainer.expected_value),
    'metrics': {
        'pm25': {'r2': round(r2_pm25,4), 'mae': round(mae_pm25,2), 'rmse': round(rmse_pm25,2)},
        'aqi':  {'r2': round(r2_aqi,4),  'mae': round(mae_aqi,2),  'rmse': round(rmse_aqi,2)},
    },
    'train_range': f'{df_train.index.min().date()} to {df_train.index.max().date()}',
    'test_range':  f'{df_test.index.min().date()} to {df_test.index.max().date()}',
    'n_train': len(df_train), 'n_test': len(df_test),
}
with open('model_output/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

In [ ]:
# Download everything
from google.colab import files
for path in ['model_output/xgb_pm25.joblib', 'model_output/xgb_aqi.joblib',
             'model_output/model_metadata.json', 'model_evaluation.png', 'shap_summary.png']:
    files.download(path)

print('\n✅ Done! Place the 3 model files here in your project:')
print('   vayu-sangam/backend/data/ml/xgb_pm25.joblib')
print('   vayu-sangam/backend/data/ml/xgb_aqi.joblib')
print('   vayu-sangam/backend/data/ml/model_metadata.json')

---
## ✅ After Training — What to do next

1. Place the 3 downloaded files in `vayu-sangam/backend/data/ml/`
2. Tell Antigravity to wire them into the API (`/api/ml/shap` endpoint + SHAP explainability)

### AFNO-v4-pm25 (benchmark reference for your PPT)
- **What:** Attention-based Fourier Neural Operator trained on 2016 WRF-Chem India data
- **License:** Apache 2.0 — https://huggingface.co/NSCLIMATE/AFNO-v4-pm25
- **Why not used directly:** Needs WRF-Chem spatial grid input — not compatible with station observations
- **Use:** Cite as related work / production-grade baseline in your presentation